## Load Packages and Dataset

In [1]:
import pandas as pd

impound = pd.read_csv("https://raw.githubusercontent.com/srajabia/DATA205-Final-Project/refs/heads/main/data/raw/animal_impound.csv")
outcome = pd.read_csv("https://raw.githubusercontent.com/srajabia/DATA205-Final-Project/refs/heads/main/data/raw/animal_outcome.csv")

## Clean column names

In [2]:
impound.columns = impound.columns.str.lower()
impound.columns = impound.columns.str.replace(" ", "_")
outcome.columns = outcome.columns.str.lower()
outcome.columns = outcome.columns.str.replace(" ", "_")

## Join datasets

In [3]:
animals = pd.merge(impound, outcome, how="inner", on="impound_number")
animals_original = pd.merge(impound, outcome, how="inner", on="impound_number")

In [4]:
# check if duplicate columns contain the same amount of usable information 
(animals.count() / len(animals)).sort_values(ascending=True)

kennel_substatus                  0.089484
bite_history                      0.152882
pet's_age_in_months               0.273032
animal_source_x                   0.309427
reason                            0.319448
pet's_age_in_years                0.521913
geo_location                      0.580104
longitude                         0.580104
latitude                          0.580104
tag_number                        0.738559
jurisdiction_y                    0.922028
jurisdiction_x                    0.922028
pet's_date_of_birth               0.922211
outcome_condition                 0.946859
animal_outcome_condition          0.946859
outcome_date_and_time             0.995264
departure_date_and_time           0.995264
animal_size                       0.998587
microchip_scan_info               0.999974
outcome_subtype_x                 1.000000
intake_condition                  1.000000
animal_identification_number_x    1.000000
location                          1.000000
fostered   

## drop duplicate columns

In [5]:
animals = (animals
           .drop(columns=["animal_identification_number_y", 
                          "animal_source_x", 
                          "jurisdiction_y", 
                          "outcome_type_y", 
                          "outcome_subtype_y",
                          "impound_date_and_time",
                          "departure_date_and_time",
                          "release_date",
                          "stamp",
                          "animal_outcome_condition",
                          "impound_type",
                          "intake_subcategory"])
           .rename(columns={"animal_identification_number_x": "animal_identification_number",
                            "animal_source_y": "animal_source",
                            "jurisdiction_x": "jurisdiction",
                            "outcome_type_x": "outcome_type",
                            "outcome_subtype_x": "outcome_subtype"
                           })
           )

In [6]:
# determine column usability based on % usable observations
(animals.count() / len(animals)).sort_values(ascending=True)

kennel_substatus                0.089484
bite_history                    0.152882
pet's_age_in_months             0.273032
reason                          0.319448
pet's_age_in_years              0.521913
longitude                       0.580104
latitude                        0.580104
geo_location                    0.580104
tag_number                      0.738559
jurisdiction                    0.922028
pet's_date_of_birth             0.922211
outcome_condition               0.946859
outcome_date_and_time           0.995264
animal_size                     0.998587
microchip_scan_info             0.999974
kennel_status                   1.000000
outcome_type                    1.000000
outcome_subtype                 1.000000
intake_date_and_time            1.000000
intake_condition                1.000000
impound_number                  1.000000
animal_identification_number    1.000000
intake_type                     1.000000
intake_subtype                  1.000000
animal_type     

## Remove reduntant and unusable columns

In [7]:
animals = animals.drop(columns=["kennel_substatus",
                                "pet's_age_in_months",
                                "pet's_age_in_years",
                                "reason",
                                "geo_location",
                                "tag_number",
                                "jurisdiction",
                                "location",
                                "kennel_number/location",
                                "pet's_age"])

## Correct NA values

In [8]:
animals = animals.replace({
    "sex_of_animal": {"U": pd.NA},
    "bite_history": {"U": pd.NA, pd.NA: "N"},
    "microchip_scan_info": {"UNABLE TO SCAN": pd.NA},
    "animal_size": {"nan": pd.NA, "0": pd.NA},
})


## Clean observations

In [9]:
# clean animal_size observations 
animals["animal_size"].unique() 
animals["animal_size"] = animals["animal_size"].replace({
    "TOY": "SMALL",
    "KITTE": "SMALL",
    "PUPPY": "SMALL",
    "X-LRG": "LARGE",
    "M": "MED"
})

## correct data types

In [10]:
print(animals.dtypes)

# convert animal_size data type to factors 
animals["animal_size"] = (animals["animal_size"]
                          .astype("category")
                          .cat.reorder_categories(["SMALL", "MED", "LARGE"], ordered=True)
                          ) 

# convert date of birth to datetime
animals["pet's_date_of_birth"] = pd.to_datetime(animals["pet's_date_of_birth"], format = "%Y/%m/%d %H:%M:%S.%f")

# convert intake date and time to datetime
animals["intake_date_and_time"] = pd.to_datetime(animals["intake_date_and_time"], format = "%m/%d/%Y %I:%M:%S %p")

# convert outcome date to datetime
animals["outcome_date_and_time"] = pd.to_datetime(animals["outcome_date_and_time"], format = "%m/%d/%Y %I:%M:%S %p")

# convert fostered and bite history to boolean
animals["fostered"] = (animals["fostered"]
                       .replace({"Y": True, "N": False})
                       .astype("boolean")
                       )
animals["bite_history"] = (animals["bite_history"]
                          .replace({"Y": True, "N": False})
                          .astype("boolean")
                          )

impound_number                      str
kennel_status                       str
animal_identification_number        str
microchip_scan_info                 str
intake_type                         str
intake_subtype                      str
intake_condition                    str
intake_date_and_time                str
outcome_type                        str
outcome_subtype                     str
outcome_condition                   str
outcome_date_and_time               str
fostered                            str
latitude                        float64
longitude                       float64
animal_source                       str
animal_type                         str
sex_of_animal                       str
pet's_date_of_birth                 str
bite_history                        str
animal_size                         str
color_of_animal                     str
breed_of_animal                     str
dtype: object


## create useful columns 

In [11]:
# add bool column "fixed"
animals["fixed"] = animals["sex_of_animal"].apply(lambda x: True if x == "S" or x == "N" else pd.NA if pd.isna(x) else False)
animals["fixed"] = animals["fixed"].astype("boolean")
# change sex back to F or M
animals["sex_of_animal"] = animals["sex_of_animal"].replace({"S":"F", "N": "M"})

# add length_of_stay column
animals["stay_in_days"] = (
    (animals["outcome_date_and_time"] - animals["intake_date_and_time"])
    .dt.total_seconds() / 86400).round(0)
# mark negative stays as NA
animals.loc[animals["stay_in_days"] < 0, ["stay_in_days", "outcome_date_and_time"]] = pd.NA
# remove negative signs from 0 day stays
animals["stay_in_days"] = animals["stay_in_days"].abs()

# add age column
animals["age_days"] = (
    (animals["intake_date_and_time"] - animals["pet's_date_of_birth"])
    .dt.total_seconds() / 86400).round(0)
# mark negative ages as NA
animals.loc[animals["age_days"] < 0, ["age_days", "pet's_date_of_birth"]] = pd.NA
# remove negative signs from those at age 0
animals["age_days"] = animals["age_days"].abs()

## standardize values

In [21]:
# Split animal breeds
breeds = animals["breed_of_animal"].str.split("/", n=1, expand=True)
animals["primary_breed"] = breeds[0].str.strip()
animals["secondary_breed"] = breeds[1].str.strip()
animals["mixed_breed"] = animals["secondary_breed"].apply(lambda x: False if pd.isna(x) else True)

# Split animal colors
colors = animals["color_of_animal"].str.split("/", n=1, expand=True)
animals["primary_color"] = colors[0].str.strip()
animals["secondary_color"] = colors[1].str.strip()
animals["multicolored"] = animals["secondary_color"].apply(lambda x: False if pd.isna(x) else True)

## reorder columns

In [14]:
animals = animals[["impound_number", 
                   "animal_identification_number", 
                   "animal_source", 
                   "animal_type",
                   "breed_of_animal",
                   "color_of_animal",
                   "animal_size",
                   "sex_of_animal",
                   "fixed",
                   "age_days",
                   "pet's_date_of_birth",
                   "bite_history",
                   "microchip_scan_info",
                   "kennel_status",
                   "intake_type",
                   "intake_subtype",
                   "intake_condition",
                   "outcome_type",
                   "outcome_subtype",
                   "outcome_condition",
                   "intake_date_and_time",
                   "outcome_date_and_time",
                   "stay_in_days",                   
                   "fostered",
                   "latitude",
                   "longitude"]]